# Báo Cáo Thực Nghiệm: Mạng Tích Chập 1D (1D CNN) Văn Bản Bằng TensorFlow / Keras
## Bài Toán: Phân Loại Cảm Xúc Bình Luận Khách Hàng (NLP Sentiment Classification)
**Học phần:** Phát triển Hệ thống Thông minh (Intelligent System Development)  
**Nhiệm vụ:** Assignment 04 - Ánh xạ kiến trúc 1D CNN văn bản sang TensorFlow (`layers.Embedding`, `layers.Conv1D`)  
**Dữ liệu:** `Womens Clothing E-Commerce Reviews.csv` (23,486 đánh giá)


In [1]:
import os
os.environ["MPLBACKEND"] = "Agg"
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
import re
import time
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Phiên bản TensorFlow: {tf.__version__}")


Phiên bản TensorFlow: 2.21.0


### 1. Đọc & Tiền Xử Lý Dữ Liệu Đồng Bộ


In [2]:
DATA_PATH = "../../../Assigment 02/customer_behavior/data/Womens Clothing E-Commerce Reviews.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "src/Assigment 02/customer_behavior/data/Womens Clothing E-Commerce Reviews.csv"

df_raw = pd.read_csv(DATA_PATH)
df_clean = df_raw.dropna(subset=['Review Text', 'Recommended IND']).copy()
df_clean['Recommended IND'] = df_clean['Recommended IND'].astype(int)

def clean_text(text):
    return re.findall(r'[a-zA-Z]+', str(text).lower())

tokenized_texts = [clean_text(t) for t in df_clean['Review Text']]
word_counts = Counter(word for doc in tokenized_texts for word in doc)
VOCAB_SIZE = 3000
MAX_LEN = 50

most_common = word_counts.most_common(VOCAB_SIZE - 2)
word2idx = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1

def text_to_sequence(tokens, max_len=MAX_LEN):
    seq = [word2idx.get(w, 1) for w in tokens[:max_len]]
    if len(seq) < max_len:
        seq = seq + [0] * (max_len - len(seq))
    return seq

X_sequences = np.array([text_to_sequence(t) for t in tokenized_texts], dtype=np.int32)
y_labels = df_clean['Recommended IND'].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X_sequences, y_labels, test_size=0.20, random_state=RANDOM_SEED, stratify=y_labels
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")


X_train shape: (18112, 50), y_train shape: (18112,)


### 2. Định Nghĩa Mô Hình 1D CNN Văn Bản Bằng TensorFlow
- `layers.Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LEN)`
- `layers.Conv1D(filters=32, kernel_size=3, activation='relu')`
- `layers.MaxPooling1D(pool_size=2)`
- `layers.Flatten()`
- `layers.Dense(16, activation='relu')`
- `layers.Dense(1, activation='sigmoid')`


In [3]:
model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=32),
    layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.002),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 32)         │        96,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 48, 32)         │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │        12,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,425 (435.25 KB)

 Trainable params: 111,425 (435.25 KB)

 Non-trainable params: 0 (0.00 B)

### 3. Huấn Luyện Mô Hình Bằng TensorFlow


In [4]:
epochs = 8
batch_size = 128

start_time = time.time()
print("Bắt đầu huấn luyện mô hình 1D CNN Văn bản TensorFlow...")

fit_history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    verbose=1
)

train_time = time.time() - start_time
print(f"\nHuấn luyện TensorFlow hoàn tất trong: {train_time:.2f} giây!")


Bắt đầu huấn luyện mô hình 1D CNN Văn bản TensorFlow...
Epoch 1/8


  1/142 ━━━━━━━━━━━━━━━━━━━━ 1:55 821ms/step - accuracy: 0.1797 - loss: 0.7090

 19/142 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7788 - loss: 0.5503    

 36/142 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8023 - loss: 0.5008

 51/142 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8091 - loss: 0.4780

 66/142 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8144 - loss: 0.4563

... (107 outputs left out)

124/142 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9905 - loss: 0.0300

139/142 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9912 - loss: 0.0284

142/142 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9914 - loss: 0.0279 - val_accuracy: 0.8664 - val_loss: 0.6657



Huấn luyện TensorFlow hoàn tất trong: 6.89 giây!


### 4. Đánh Giá Toàn Diện & Trực Quan Hóa Kết Quả


In [5]:
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_prob)

print(f"=== KẾT QUẢ ĐÁNH GIÁ 1D CNN VĂN BẢN (TENSORFLOW) ===")
print(f"• Accuracy : {acc:.4f}")
print(f"• Precision: {prec:.4f}")
print(f"• Recall   : {rec:.4f}")
print(f"• F1-Score : {f1:.4f}")
print(f"• ROC-AUC  : {auc:.4f}")
print(f"• Thời gian: {train_time:.2f} giây")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, epochs + 1), fit_history.history["loss"], 'o-', color='#f59e0b', label='Train Loss')
axes[0].plot(range(1, epochs + 1), fit_history.history["val_loss"], 's--', color='#ef4444', label='Val Loss')
axes[0].set_title('TensorFlow: Đường Cong Hàm Mất Mát BCE', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].legend()

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Oranges', ax=axes[1], cbar=False)
axes[1].set_title('TensorFlow: Ma Trận Nhầm Lẫn Cảm Xúc', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nhãn Dự Đoán')
axes[1].set_ylabel('Nhãn Thực Tế')

plt.tight_layout()
os.makedirs("../reports/figures", exist_ok=True)
plt.savefig("../reports/figures/03_comments_tensorflow_results.png", dpi=300)
plt.show()


  1/142 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step

103/142 ━━━━━━━━━━━━━━━━━━━━ 0s 494us/step

142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 727us/step

142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 779us/step


=== KẾT QUẢ ĐÁNH GIÁ 1D CNN VĂN BẢN (TENSORFLOW) ===
• Accuracy : 0.8664
• Precision: 0.8957
• Recall   : 0.9472
• F1-Score : 0.9207
• ROC-AUC  : 0.8669
• Thời gian: 6.89 giây


/var/folders/x0/279_kvcj3dvbd1fkpvt_8j7r0000gn/T/ipykernel_4108/2921739948.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Lưu Trữ Kết Quả Thực Nghiệm & Tổng Hợp Đối Chuẩn


In [6]:
results_df = pd.DataFrame([{
    "Framework": "TensorFlow",
    "Dataset": "Customer Comments (23.5k)",
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1": f1,
    "ROC_AUC": auc,
    "Train_Time_Sec": train_time,
    "Epochs": epochs,
    "Batch_Size": batch_size
}])

results_df.to_csv("../reports/model_results_tensorflow.csv", index=False)
print("Đã lưu kết quả thành công vào ../reports/model_results_tensorflow.csv")

dfs = []
for fname in ["model_results_numpy.csv", "model_results_pytorch.csv", "model_results_tensorflow.csv"]:
    p = os.path.join("../reports", fname)
    if os.path.exists(p):
        dfs.append(pd.read_csv(p))

if dfs:
    summary_df = pd.concat(dfs, ignore_index=True)
    summary_df.to_csv("../reports/model_results.csv", index=False)
    print("\n=== BẢNG TỔNG HỢP ĐỐI CHUẨN 3 FRAMEWORK TRÊN TẬP COMMENTS ===")
    display(summary_df)


Đã lưu kết quả thành công vào ../reports/model_results_tensorflow.csv

=== BẢNG TỔNG HỢP ĐỐI CHUẨN 3 FRAMEWORK TRÊN TẬP COMMENTS ===


,Framework,Dataset,Accuracy,Precision,Recall,F1,ROC_AUC,Train_Time_Sec,Epochs,Batch_Size
0,NumPy From Scratch,Customer Comments (23.5k),0.875469,0.904762,0.947695,0.925731,0.874306,198.896814,8,128
1,PyTorch,Customer Comments (23.5k),0.855376,0.906117,0.918576,0.912304,0.875471,9.998415,8,128
2,TensorFlow,Customer Comments (23.5k),0.866416,0.895716,0.947156,0.920718,0.866923,6.889872,8,128
